# 02 — RL Agent Training

Trains 4 agents per ticker (PPO and DQN × price-only and price+sentiment):

| Agent | State |
|-------|-------|
| PPO   | price-only |
| PPO   | price + sentiment |
| DQN   | price-only |
| DQN   | price + sentiment |

**Note:** SAC requires a continuous action space — DQN is the correct off-policy algorithm for our discrete (reduce/hold/increase) action space.

**Train period:** Oct 1 – Nov 30, 2025 (43 trading days)
**Test period:** Dec 1 – Dec 31, 2025

Models saved to `models/`.

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from trading_env import TradingEnv, make_envs

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

MODELS_DIR = PROJECT_ROOT / 'models'
DATA_DIR   = PROJECT_ROOT / 'data'
MODELS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. Load Dataset

In [2]:
daily = pd.read_csv(DATA_DIR / 'daily_features.csv', parse_dates=['date'])

TICKERS = ['NVDA', 'GOOG', 'TSLA']
TRAIN_CUTOFF = '2025-11-30'

print('Dataset shape:', daily.shape)
print('Tickers:', daily['ticker'].unique())
print()

for t in TICKERS:
    sub = daily[daily['ticker'] == t]
    train = sub[sub['date'] <= TRAIN_CUTOFF]
    test  = sub[sub['date'] >  TRAIN_CUTOFF]
    print(f'{t}: {len(train)} train days, {len(test)} test days')

Dataset shape: (192, 33)
Tickers: ['GOOG' 'NVDA' 'TSLA']

NVDA: 42 train days, 22 test days
GOOG: 42 train days, 22 test days
TSLA: 42 train days, 22 test days


## 2. Sanity Check — Environment

In [ ]:
# Keep penalties at zero for the main feature ablation.
# Raise these in a separate reward-design experiment.
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.0,
    
    # added volatility threshold
    use_volatility_gate = True,
    vol_threshold_quantile = 0.3,
)

# Quick environment checks on NVDA
envs = make_envs(daily, 'NVDA', TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

for env_key, label in [
    ('train_price', 'price-only env'),
    ('train_sentiment_basic', 'basic sentiment env'),
]:
    print(f'Checking {label}...')
    check_env(envs[env_key], warn=True)
    print('OK')

print(f"Obs dim (price-only):       {envs['train_price'].observation_space.shape[0]}")
print(f"Obs dim (sentiment_basic):  {envs['train_sentiment_basic'].observation_space.shape[0]}")


Checking price-only env...
OK
Checking basic sentiment env...
OK
Obs dim (price-only):       10
Obs dim (sentiment_basic):  17


## 3. Training Configuration

In [ ]:
# 30k timesteps is sufficient for our 43-day training window.
# Each episode is ~43 steps, so this is ~700 full episodes per agent.
# Increase to 100k+ if you want longer convergence runs (takes ~5 min/model).
TOTAL_TIMESTEPS = 30_000

# Keep penalties at zero for the main feature ablation.
# Raise these in a separate reward-design experiment.
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.0,
    
    # added volatility threshold
    use_volatility_gate = True,
    vol_threshold_quantile = 0.3,
)

PPO_KWARGS = dict(
    policy='MlpPolicy',
    n_steps=43,         # one full episode per rollout
    batch_size=43,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=3e-4,
    verbose=0,
)

DQN_KWARGS = dict(
    policy='MlpPolicy',
    learning_rate=1e-3,
    buffer_size=5_000,
    learning_starts=43,
    batch_size=32,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    exploration_final_eps=0.05,
    exploration_fraction=0.3,
    verbose=0,
)

print(f'Training {TOTAL_TIMESTEPS:,} timesteps per agent.')
print('Environment kwargs:', ENV_KWARGS)
print('Configs set.')


Training 30,000 timesteps per agent.
Environment kwargs: {'reward_mode': 'log_return', 'drawdown_penalty': 0.0, 'volatility_penalty': 0.0}
Configs set.


## 4. Train All Agents

This trains 4 agents × 3 tickers = **12 models** total. At 30k timesteps each model takes ~50s (PPO) or ~30s (DQN).

In [5]:
from tqdm.notebook import tqdm

trained_models = {}  # key: (ticker, algo, state_type)

configs = [
    ('PPO', 'price',           PPO, PPO_KWARGS, 'train_price'),
    ('PPO', 'sentiment_basic', PPO, PPO_KWARGS, 'train_sentiment_basic'),
    ('DQN', 'price',           DQN, DQN_KWARGS, 'train_price'),
    ('DQN', 'sentiment_basic', DQN, DQN_KWARGS, 'train_sentiment_basic'),
]

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

    for algo_name, state_type, AlgoClass, kwargs, env_key in tqdm(configs, desc=ticker):
        env = Monitor(envs[env_key])
        model = AlgoClass(env=env, seed=42, **kwargs)
        model.learn(total_timesteps=TOTAL_TIMESTEPS)

        key = (ticker, algo_name, state_type)
        trained_models[key] = model

        save_path = MODELS_DIR / f'{ticker}_{algo_name}_{state_type}'
        model.save(str(save_path))
        print(f'  Saved {save_path.name}')

print('\nAll models trained.')



=== NVDA ===


NVDA:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved NVDA_PPO_price
  Saved NVDA_PPO_sentiment_basic
  Saved NVDA_DQN_price
  Saved NVDA_DQN_sentiment_basic

=== GOOG ===


GOOG:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved GOOG_PPO_price
  Saved GOOG_PPO_sentiment_basic
  Saved GOOG_DQN_price
  Saved GOOG_DQN_sentiment_basic

=== TSLA ===


TSLA:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved TSLA_PPO_price
  Saved TSLA_PPO_sentiment_basic
  Saved TSLA_DQN_price
  Saved TSLA_DQN_sentiment_basic

All models trained.


## 5. Quick Training Sanity Check

Run each model on its test set and print final portfolio value.

In [6]:
def evaluate_model(model, env):
    """Run one episode and return history + final portfolio value."""
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
    return env.get_history(), info['portfolio_value']

print(f'Initial cash: $10,000.00\n')
print(f'{"Ticker":<6} {"Algo":<5} {"State":<20} {"Final $":>10}')
print('-' * 48)

test_results = {}
for ticker in TICKERS:
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)
    for algo_name, state_type, AlgoClass, _, env_key in configs:
        test_env_key = env_key.replace('train_', 'test_')
        key = (ticker, algo_name, state_type)
        model = trained_models[key]
        history, final_val = evaluate_model(model, envs[test_env_key])
        test_results[key] = (history, final_val)
        print(f'{ticker:<6} {algo_name:<5} {state_type:<20} ${final_val:>9.2f}')


Initial cash: $10,000.00

Ticker Algo  State                   Final $
------------------------------------------------
NVDA   PPO   price                $  9880.56
NVDA   PPO   sentiment_basic      $ 10121.82
NVDA   DQN   price                $ 10296.39
NVDA   DQN   sentiment_basic      $ 10418.89
GOOG   PPO   price                $ 10101.01
GOOG   PPO   sentiment_basic      $ 10080.34
GOOG   DQN   price                $  9813.22
GOOG   DQN   sentiment_basic      $ 10016.68
TSLA   PPO   price                $ 10544.98
TSLA   PPO   sentiment_basic      $ 10438.33
TSLA   DQN   price                $ 10446.36
TSLA   DQN   sentiment_basic      $  9940.93


## 6. Short-Selling Extension

Trains 4 additional agents per ticker with an expanded 4-action space:

| Action | Meaning |
|--------|---------|
| 0 | Close all — sell longs, cover shorts |
| 1 | Hold |
| 2 | Buy long (covers shorts first) |
| 3 | Short (closes longs first) |

This allows the agent to profit on bad-sentiment days rather than only moving to cash.
12 new models total (PPO/DQN × price/sentiment × 3 tickers), saved as `{TICKER}_{ALGO}_{state}_short.zip`.

In [7]:
configs_short = [
    ('PPO', 'price_short',           PPO, PPO_KWARGS, 'train_price_short'),
    ('PPO', 'sentiment_basic_short', PPO, PPO_KWARGS, 'train_sentiment_basic_short'),
    ('DQN', 'price_short',           DQN, DQN_KWARGS, 'train_price_short'),
    ('DQN', 'sentiment_basic_short', DQN, DQN_KWARGS, 'train_sentiment_basic_short'),
]

for ticker in TICKERS:
    print(f'\n=== {ticker} (short-enabled) ===')
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS, allow_short=True)

    for algo_name, state_type, AlgoClass, kwargs, env_key in tqdm(configs_short, desc=ticker):
        env = Monitor(envs[env_key])
        model = AlgoClass(env=env, seed=42, **kwargs)
        model.learn(total_timesteps=TOTAL_TIMESTEPS)

        key = (ticker, algo_name, state_type)
        trained_models[key] = model

        save_path = MODELS_DIR / f'{ticker}_{algo_name}_{state_type}'
        model.save(str(save_path))
        print(f'  Saved {save_path.name}')

print('\nAll short-enabled models trained.')


=== NVDA (short-enabled) ===


NVDA:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved NVDA_PPO_price_short
  Saved NVDA_PPO_sentiment_basic_short
  Saved NVDA_DQN_price_short
  Saved NVDA_DQN_sentiment_basic_short

=== GOOG (short-enabled) ===


GOOG:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved GOOG_PPO_price_short
  Saved GOOG_PPO_sentiment_basic_short
  Saved GOOG_DQN_price_short
  Saved GOOG_DQN_sentiment_basic_short

=== TSLA (short-enabled) ===


TSLA:   0%|          | 0/4 [00:00<?, ?it/s]

  Saved TSLA_PPO_price_short
  Saved TSLA_PPO_sentiment_basic_short
  Saved TSLA_DQN_price_short
  Saved TSLA_DQN_sentiment_basic_short

All short-enabled models trained.


In [8]:
print(f'Initial cash: $10,000.00\n')
print(f'{"Ticker":<6} {"Algo":<5} {"State":<26} {"Final $":>10}  {"Short days":>11}')
print('-' * 62)

for ticker in TICKERS:
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS, allow_short=True)
    for algo_name, state_type, _, _, env_key in configs_short:
        test_env_key = env_key.replace('train_', 'test_')
        key = (ticker, algo_name, state_type)
        model = trained_models[key]
        history, final_val = evaluate_model(model, envs[test_env_key])
        short_days = int((history['action'] == 3).sum())
        print(f'{ticker:<6} {algo_name:<5} {state_type:<26} ${final_val:>9.2f}  {short_days:>11}')

Initial cash: $10,000.00

Ticker Algo  State                         Final $   Short days
--------------------------------------------------------------
NVDA   PPO   price_short                $  9821.05            5
NVDA   PPO   sentiment_basic_short      $ 10063.87            8
NVDA   DQN   price_short                $ 10094.66           10
NVDA   DQN   sentiment_basic_short      $  9878.45           12
GOOG   PPO   price_short                $ 10229.44            8
GOOG   PPO   sentiment_basic_short      $  9984.72            8
GOOG   DQN   price_short                $  9679.62           13
GOOG   DQN   sentiment_basic_short      $  9927.36            1
TSLA   PPO   price_short                $ 11156.50            7
TSLA   PPO   sentiment_basic_short      $  9672.29           11
TSLA   DQN   price_short                $ 10460.31           12
TSLA   DQN   sentiment_basic_short      $ 10218.71            9
